In [1]:
# =============================================
#                   IMPORT
# =============================================
from keras.optimizers                   import Adam
from keras.callbacks                    import EarlyStopping, ReduceLROnPlateau
from preProcessingData                  import PreProcessData
# from model.systemModel                  import SystemModel as systemModel
from model.componentModel               import SystemModel as componentModel
from sklearn.preprocessing              import MinMaxScaler

from tqdm                               import tqdm
from datetime                           import datetime
from prettytable                        import PrettyTable
import plotly.graph_objects             as go
import tensorflow                       as tf
import numpy                            as np 
import pandas                           as pd
import matplotlib.pyplot                as plt
import os, warnings
import random
import json


# ====================================
# Set debug level and 
# ------------------------------------
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"  
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'  # or any {'0', '1', '2'}
warnings.filterwarnings('ignore') 

np.random.seed(1)
random.seed(1)
tf.random.set_seed(1)

# ====================================
# Config
# ------------------------------------
with open('utility/modelConfig.json', 'r') as configFile:
    config = json.load(configFile)
myConfig   = config["system"]

2024-12-16 21:28:20.087031: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-12-16 21:28:20.234471: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2024-12-16 21:28:20.234486: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2024-12-16 21:28:20.258916: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-16 21:28:21.213229: W tensorflow/stream_executor/platform/de

##### HI COMPONENT AND SYSTEM

In [2]:
preData                 = PreProcessData(config = myConfig)                    # Preprocessing data

cr1, cr2, cv2           = preData.loadH5File(name = "WITHOUTNonCriticalData", keys = ["cr1", "cr2", "cv2"])                           # Load the data from critical file



# --------------------------------
# Add hour column into the file
# print(cv2.columns)
withHourCr1             = preData.addHours(cr1)
withHourCr2             = preData.addHours(cr2)
withHourCv2             = preData.addHours(cv2)


# --------------------------------
# Use for system HI
# --------------------------------
insertedCr1Sys, insertedCr2Sys,  insertedCv2Sys     = preData.insertForTheSameLength(data         = [withHourCr1.copy(),withHourCr2.copy(),withHourCv2.copy()], 
                                                                                    validColumn  = [preData.crusher1Value, preData.crusher2Value, preData.conveyorValue + preData.crusher2ConveyorValue])

withIDCr1Sys, withIDCr2Sys, withIDCv2Sys, IdSysSys  = preData.attachIDPerEvent(df = [insertedCr1Sys,insertedCr2Sys,insertedCv2Sys], mainternanceAsCriticalSd = True)

insertedCr1 = withHourCr1
insertedCr2 = withHourCr2
insertedCv2 = withHourCv2

withIDCr1, withIDCr2, withIDCv2, IdSys = preData.attachIDPerEvent(df = [insertedCr1,insertedCr2,insertedCv2], mainternanceAsCriticalSd = True)


addedData               = preData.addition_var(withIDCr1) 


handledValues           = preData.crusher1Value + preData.addEfficiencyCr1 + preData.addedManipulatedCr1
# -----------------------------
# Normalize
normalizedCr1           = preData.normalizeBuilinFunction(addedData, headline = handledValues)  
normalizedCr2           = preData.normalizeBuilinFunction(withIDCr2, headline = preData.crusher2Value)  
normalizedCv2           = preData.normalizeBuilinFunction(withIDCv2, headline = preData.conveyorValue + preData.crusher2Value)  

# -----------------------------
# Add the Cycle and RUL to 
# each component
withTimeDataCr1         = preData.addCycleAndRul(normalizedCr1)                        # Add the time to the data
withTimeDataCr2         = preData.addCycleAndRul(normalizedCr2)                        # Add the time to the data
withTimeDataCv2         = preData.addCycleAndRul(normalizedCv2)                        # Add the time to the data

# -----------------------------
# Add the cycle and RUL of system
systemData                  = pd.DataFrame()
systemData['hours']         = withTimeDataCr1["hours"]
systemData['trajectoryID']  = IdSys
withTimeDataSys             = preData.addCycleAndRul(systemData)

# -----------------------------
# Extract before splitting
dataCr1                 = withTimeDataCr1[handledValues]
cycleCr1                = withTimeDataCr1["cycle"]
rulCr1                  = withTimeDataCr1["rul"]
timeCr1                 = withTimeDataCr1["hours"]
trajectIDCr1            = withTimeDataCr1["trajectoryID"]

dataCr2                 = withTimeDataCr2[preData.crusher2Value]
cycleCr2                = withTimeDataCr2["cycle"]
rulCr2                  = withTimeDataCr2["rul"]
timeCr2                 = withTimeDataCr2["hours"]
trajectIDCr2            = withTimeDataCr2["trajectoryID"]

dataCv2                 = withTimeDataCv2[preData.conveyorValue + preData.crusher2ConveyorValue]
cycleCv2                = withTimeDataCv2["cycle"]
rulCv2                  = withTimeDataCv2["rul"]
timeCv2                 = withTimeDataCv2["hours"]
trajectIDCv2            = withTimeDataCv2["trajectoryID"]

systemRUL               = withTimeDataSys["rul"]


print(timeCr1)
print(timeCr2)
print(timeCv2)
print(insertedCr1Sys["hours"])



Process:30512||2024-12-16 21:28:22,737-INFO [preProcessingData.py:loadH5File():590]:Loading H5 file ...
Process:30512||2024-12-16 21:28:23,896-INFO [preProcessingData.py:loadH5File():596]:Finish loading data from files ...
Process:30512||2024-12-16 21:28:27,674-INFO [preProcessingData.py:attachIDPerEvent():437]:Attaching ID into trajectories
Process:30512||2024-12-16 21:28:46,562-INFO [preProcessingData.py:attachIDPerEvent():437]:Attaching ID into trajectories
Process:30512||2024-12-16 21:29:02,977-INFO [preProcessingData.py:normalizeBuilinFunction():1454]:Normalizing the data ....
Process:30512||2024-12-16 21:29:03,487-INFO [preProcessingData.py:normalizeBuilinFunction():1454]:Normalizing the data ....
Process:30512||2024-12-16 21:29:03,929-INFO [preProcessingData.py:normalizeBuilinFunction():1454]:Normalizing the data ....


0            0.000
1            0.017
2            0.033
3            0.050
4            0.067
            ...   
321370    8759.933
321371    8759.950
321372    8759.967
321373    8759.983
321374    8760.000
Name: hours, Length: 321375, dtype: float64
0            0.000
1            0.017
2            0.033
3            0.050
4            0.067
            ...   
466645    8759.933
466646    8759.950
466647    8759.967
466648    8759.983
466649    8760.000
Name: hours, Length: 466650, dtype: float64
0            0.000
1            0.017
2            0.033
3            0.050
4            0.067
            ...   
451063    8759.933
451064    8759.950
451065    8759.967
451066    8759.983
451067    8760.000
Name: hours, Length: 451068, dtype: float64
0            0.000
1            0.017
2            0.033
3            0.050
4            0.067
            ...   
494190    8759.933
494191    8759.950
494192    8759.967
494193    8759.983
494194    8760.000
Name: hours, Length: 494195, dty

###### Crusher1

In [3]:
def calculate_mae_for_numpy(list1, list2):
    if len(list1) != len(list2):
        raise ValueError("The two lists must have the same length.")
    
    maes = []  # To store the MAE for each pair of arrays
    for arr1, arr2 in zip(list1, list2):
        if arr1.shape != arr2.shape:
            raise ValueError("Each pair of arrays must have the same shape.")
        mae = np.mean(np.abs(arr1 - arr2))  # Element-wise absolute difference, then mean
        maes.append(mae)
    
    overall_mae = np.mean(maes)  # Average MAE across all pairs
    return overall_mae

In [4]:
myConfig["component"] = "Cr1"

model       = componentModel(myConfig) 
model.build(input_shape=(None, myConfig["seqLength"], myConfig["inCr1"]))  # Batch size is None, feature size is 32()
model.summary()
# model.load_weights(model_dir = "checkpoint/bestModel37/")
model.load_weights(model_dir = "checkpoint/crusher1Best_Trendability/")

The shape of sysHILayer (None, 4, 5)


2024-12-16 21:29:04.810553: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-12-16 21:29:04.810830: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-12-16 21:29:04.811092: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2024-12-16 21:29:04.811185: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublas.so.11'; dlerror: libcublas.so.11: cannot open shared object file: No such file or directory
2024-12-16 21:29:04.811265: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not lo

The shape of sysHILayer (None, 4, 5)
Model: "system_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sensor_selection (SensorSel  multiple                 0 (unused)
 ection)                                                         
                                                                 
 encoder (Functional)        [(None, 1),               122306    
                              (None, 4, 5)]                      
                                                                 
 decoder (Functional)        (None, 64, 41)            121941    
                                                                 
 hi_system_and_rul (HiSystem  multiple                 6413      
 AndRul)                                                         
                                                                 
Total params: 250,712
Trainable params: 250,701
Non-trainable params: 11
__________

Process:30512||2024-12-16 21:29:06,011-INFO [componentModel.py:load_weights():789]:Loading the model at the direction: checkpoint/crusher1Best_Trendability/
Process:30512||2024-12-16 21:29:06,038-INFO [componentModel.py:load_weights():799]:Loaded the model at the direction:checkpoint/crusher1Best_Trendability/ successfully!!!


In [5]:
results             = {}
time                = {}
results["cr1"]      = []
results["rulCr1"]   = []
time["timeCr1"]     = []
listRUL             = []

probsAll        = []
binaryMaskAll   = []
trajectID       = withIDCr1["trajectoryID"].astype(int)
for idx in range(trajectID.max()):
    print(idx)
    count = (trajectID == (idx + 1)).sum() 
    if count <= preData.config['seqLength']:
        continue
    seqDataCr1, seqTimeCr1,seqRUL           = preData.getTensorData(dataList   =    [dataCr1, timeCr1,rulCr1], 
                                                                    trajectId  = trajectID,
                                                                    seqLength  = preData.config['seqLength'], 
                                                                    overlap    = preData.config['seqLength']-1,
                                                                    getAtIdx   = idx + 1,
                                                                    typeInput  = "listOfPandas")
    
    zCr1, binaryMaskCr1, probsMaskCr1, predictedRul     = model.getHIandRUL(seqDataCr1)
    # print(predictedRul)
    results["cr1"].append(pd.Series(tf.squeeze(zCr1).numpy()))
    results["rulCr1"].append(predictedRul)
    time["timeCr1"].append(seqTimeCr1[:,-1,:])
    listRUL.append(seqRUL[:,-1,:])


print(listRUL[0].shape, results["rulCr1"][0].shape)
mae = calculate_mae_for_numpy(listRUL, results["rulCr1"])
print("Crusher 1 MAE:", mae)

print(np.max(np.concatenate(listRUL)), np.min(np.concatenate(listRUL)))

# temp_df = pd.concat([rulCr1, rulCr2, rulCv2], axis=1)
# # print(np.max(np.concatenate(rulCr1, rulCr2 , rulCv2)), np.min(np.concatenate(rulCr1 , rulCr2, rulCv2)))
# print(temp_df.max().max(),temp_df.min().min())



Process:30512||2024-12-16 21:29:06,051-INFO [preProcessingData.py:getTensorData():1228]:Spliting data into sequences now ...


0


Process:30512||2024-12-16 21:29:06,787-INFO [preProcessingData.py:getTensorData():1228]:Spliting data into sequences now ...


1


Process:30512||2024-12-16 21:29:08,106-INFO [preProcessingData.py:getTensorData():1228]:Spliting data into sequences now ...


2
3


... (50 outputs left out)

Process:30512||2024-12-16 21:29:40,201-INFO [preProcessingData.py:getTensorData():1228]:Spliting data into sequences now ...


35


Process:30512||2024-12-16 21:29:40,473-INFO [preProcessingData.py:getTensorData():1228]:Spliting data into sequences now ...


36
(4973, 1) (4973, 1)
Crusher 1 MAE: 124.87469
667.5840000000003 0.0


In [6]:
print(time["timeCr1"][0].shape)


(4973, 1)


###### Crusher2

In [7]:
myConfig["component"] = "Cr2"

model       = componentModel(myConfig) 
model.build(input_shape=(None, myConfig["seqLength"], myConfig["inCr2"]))  # Batch size is None, feature size is 32()
model.summary()
# model.load_weights(model_dir = "checkpoint/bestModel38/")
model.load_weights(model_dir = "checkpoint/crusher2Best_Trendability/")

The shape of sysHILayer (None, 4, 5)
The shape of sysHILayer (None, 4, 5)
Model: "system_model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sensor_selection_1 (SensorS  multiple                 0 (unused)
 election)                                                       
                                                                 
 encoder (Functional)        [(None, 1),               114056    
                              (None, 4, 5)]                      
                                                                 
 decoder (Functional)        (None, 64, 30)            113680    
                                                                 
 hi_system_and_rul_1 (HiSyst  multiple                 6413      
 emAndRul)                                                       
                                                                 
Total params: 234,190
Trainable params: 234,

Process:30512||2024-12-16 21:29:43,010-INFO [componentModel.py:load_weights():789]:Loading the model at the direction: checkpoint/crusher2Best_Trendability/
Process:30512||2024-12-16 21:29:43,019-INFO [componentModel.py:load_weights():799]:Loaded the model at the direction:checkpoint/crusher2Best_Trendability/ successfully!!!


In [8]:
results["cr2"]      = []
results["rulCr2"]   = []
time["timeCr2"]     = []
listRUL             = []

probsAll        = []
binaryMaskAll   = []
trajectID       = withIDCr2["trajectoryID"].astype(int)
for idx in range(trajectID.max()):
    print(idx)
    count = (trajectID == (idx + 1)).sum() 
    if count <= preData.config['seqLength']:
        continue
    seqDataCr2, seqTimeCr2,seqRUL          = preData.getTensorData(dataList     =    [dataCr2, timeCr2, rulCr2], 
                                                                    trajectId   = trajectID,
                                                                    seqLength   = preData.config['seqLength'], 
                                                                    overlap     = preData.config['seqLength']-1,
                                                                    getAtIdx    = idx + 1,
                                                                    typeInput   = "listOfPandas")
    
    zCr2, binaryMaskCr2, probsMaskCr2, predictedRul     = model.getHIandRUL(seqDataCr2)
    # print(predictedRul)
    results["cr2"].append(pd.Series(tf.squeeze(zCr2).numpy()))
    results["rulCr2"].append(predictedRul)
    time["timeCr2"].append(seqTimeCr2[:,-1,:])
    listRUL.append(seqRUL[:,-1,:])


print(listRUL[0].shape, results["rulCr2"][0].shape)
mae = calculate_mae_for_numpy(listRUL, results["rulCr2"])
print("Crusher 2 MAE:", mae)

print(np.max(np.concatenate(listRUL)), np.min(np.concatenate(listRUL)))


Process:30512||2024-12-16 21:29:43,031-INFO [preProcessingData.py:getTensorData():1228]:Spliting data into sequences now ...


0


Process:30512||2024-12-16 21:29:43,662-INFO [preProcessingData.py:getTensorData():1228]:Spliting data into sequences now ...


1


Process:30512||2024-12-16 21:29:45,285-INFO [preProcessingData.py:getTensorData():1228]:Spliting data into sequences now ...


2


... (98 outputs left out)

Process:30512||2024-12-16 21:30:32,018-INFO [preProcessingData.py:getTensorData():1228]:Spliting data into sequences now ...


57


Process:30512||2024-12-16 21:30:33,128-INFO [preProcessingData.py:getTensorData():1228]:Spliting data into sequences now ...


58
(6083, 1) (6083, 1)
Crusher 2 MAE: 67.64371
470.3000000000002 0.0


###### Conveyorbelt 2

In [9]:
myConfig["component"] = "Cv2"

model       = componentModel(myConfig) 
model.build(input_shape=(None, myConfig["seqLength"], myConfig["inCv2"]))  # Batch size is None, feature size is 32()
model.summary()
# model.load_weights(model_dir = "checkpoint/bestModel41/")
model.load_weights(model_dir = "checkpoint/CV2Best_Trendability/")

The shape of sysHILayer (None, 4, 5)
The shape of sysHILayer (None, 4, 5)
Model: "system_model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sensor_selection_2 (SensorS  multiple                 0 (unused)
 election)                                                       
                                                                 
 encoder (Functional)        [(None, 1),               114806    
                              (None, 4, 5)]                      
                                                                 
 decoder (Functional)        (None, 64, 31)            114431    
                                                                 
 hi_system_and_rul_2 (HiSyst  multiple                 6413      
 emAndRul)                                                       
                                                                 
Total params: 235,692
Trainable params: 235,

Process:30512||2024-12-16 21:30:34,907-INFO [componentModel.py:load_weights():789]:Loading the model at the direction: checkpoint/CV2Best_Trendability/
Process:30512||2024-12-16 21:30:34,917-INFO [componentModel.py:load_weights():799]:Loaded the model at the direction:checkpoint/CV2Best_Trendability/ successfully!!!


In [10]:
results["cv2"]          = []
results["rulCv2"]       = []
time["timeCv2"]         = []
listRUL                 = []

probsAll        = []
binaryMaskAll   = []
trajectID       = withIDCv2["trajectoryID"].astype(int)
for idx in range(trajectID.max()):
    print(idx)
    count = (trajectID == (idx + 1)).sum() 
    if count <= preData.config['seqLength']:
        continue
    seqDataCv2, seqTimeCv2,seqRUL            = preData.getTensorData(dataList   =    [dataCv2, timeCv2, rulCv2], 
                                                                    trajectId  = trajectID,
                                                                    seqLength  = preData.config['seqLength'], 
                                                                    overlap    = preData.config['seqLength']-1,
                                                                    getAtIdx   = idx + 1,
                                                                    typeInput  = "listOfPandas")
    
    zCv2, binaryMaskCv2, probsMaskCv2, predictedRul     = model.getHIandRUL(seqDataCv2)
    # print(predictedRul)
    results["cv2"].append(pd.Series(tf.squeeze(zCv2).numpy()))
    results["rulCv2"].append(predictedRul)
    time["timeCv2"].append(seqTimeCv2[:,-1,:])
    listRUL.append(seqRUL[:,-1,:])


print(listRUL[0].shape, results["rulCv2"][0].shape)
mae = calculate_mae_for_numpy(listRUL, results["rulCv2"])
print("Conveyor belt 2 MAE:", mae)

print(np.max(np.concatenate(listRUL)), np.min(np.concatenate(listRUL)))


Process:30512||2024-12-16 21:30:34,930-INFO [preProcessingData.py:getTensorData():1228]:Spliting data into sequences now ...


0


Process:30512||2024-12-16 21:30:35,662-INFO [preProcessingData.py:getTensorData():1228]:Spliting data into sequences now ...


1


Process:30512||2024-12-16 21:30:37,409-INFO [preProcessingData.py:getTensorData():1228]:Spliting data into sequences now ...


2


... (112 outputs left out)

Process:30512||2024-12-16 21:31:26,890-INFO [preProcessingData.py:getTensorData():1228]:Spliting data into sequences now ...


79


Process:30512||2024-12-16 21:31:28,154-INFO [preProcessingData.py:getTensorData():1228]:Spliting data into sequences now ...


80
(5996, 1) (5996, 1)
Conveyor belt 2 MAE: 55.961838
577.2829999999994 0.0


###### System

In [11]:
# # ------------------------------
# # Normalize
# # ------------------------------
# combined_data = pd.concat(results["cr1"], ignore_index=True)
# data_reshaped = combined_data.values.reshape(-1, 1)
# scalerCr1       = MinMaxScaler()
# scalerCr1.fit(data_reshaped)
# results["cr1"]  = [pd.DataFrame(scalerCr1.transform(df.values.reshape(-1, 1))) for df in results["cr1"]]


# combined_data = pd.concat(results["cr2"], ignore_index=True)
# data_reshaped = combined_data.values.reshape(-1, 1)
# scalerCr2       = MinMaxScaler()
# scalerCr2.fit(data_reshaped)
# results["cr2"]  = [pd.DataFrame(scalerCr2.transform(df.values.reshape(-1, 1))) for df in results["cr2"]]

# combined_data = pd.concat(results["cv2"], ignore_index=True)
# data_reshaped = combined_data.values.reshape(-1, 1)
# scalerCv2       = MinMaxScaler()
# scalerCv2.fit(data_reshaped)
# results["cv2"]  = [pd.DataFrame(scalerCv2.transform(df.values.reshape(-1, 1))) for df in results["cv2"]]

In [12]:
# scalerCr1       = MinMaxScaler()
# norHI1          = scalerCr1.fit_transform(aligned_HIs[1])

# scalerCr2       = MinMaxScaler()
# norHI2          = scalerCr1.fit_transform(aligned_HIs[2])

# scalerCv2       = MinMaxScaler()
# norHI3          = scalerCv2.fit_transform(aligned_HIs[3])

In [13]:
# ------------------------------
# HI system
all_times = insertedCr1Sys["hours"]

# Create a dictionary to store aligned HIs for each component
aligned_HIs = {1: [], 2: [], 3: []}


def align_hi(HI, timeHI, all_times):
    aligned = pd.Series(index=all_times, dtype=float)       # Initialize with NaN
    for hi, times in zip(HI, timeHI):
        aligned.loc[times[:,0]] = hi.values                 # Align based on the original time
    return aligned

# Align each component's HI

aligned_HIs[1] = align_hi(results["cr1"], time["timeCr1"], all_times)
aligned_HIs[2] = align_hi(results["cr2"], time["timeCr2"], all_times)
aligned_HIs[3] = align_hi(results["cv2"], time["timeCv2"], all_times)




# Combine all aligned HIs into a DataFrame
aligned_df = pd.DataFrame({'Time': all_times,
                           'OriginalHICr1': aligned_HIs[1].values,
                           'OriginalHICr2': aligned_HIs[2].values,
                           'OriginalHICv2': aligned_HIs[3].values})

# window_size = 65
# df_moving_avg = aligned_df.iloc[:, 1:].rolling(window=window_size, min_periods=1).mean()                    # Moving average
# df_normalized = df_moving_avg.apply(lambda x: (x - x.min()) / (x.max() - x.min()), axis=0)                  # Normalize
# df_normalized = df_normalized.rename(columns={'OriginalHICr1': 'norHICr1', 'OriginalHICr2': 'norHICr2', 'OriginalHICv2': 'norHICv2'})

# # Concatenating the Time column back with the moving average
# df_result = pd.concat([aligned_df["Time"], aligned_df], axis=1)
# # Calculate the system HI (row-wise min ignoring NaN)
# df_result['System_HI'] = df_result[['norHICr1', 'norHICr2', 'norHICv2']].min(axis=1, skipna=True)


# df_result["OriginalHICr1"] = aligned_df["OriginalHICr1"]
# df_result["OriginalHICr2"] = aligned_df["OriginalHICr2"]
# df_result["OriginalHICv2"] = aligned_df["OriginalHICv2"]
aligned_df["trajectoryCr1"] = withIDCr1Sys["trajectoryID"]
aligned_df["trajectoryCr2"] = withIDCr2Sys["trajectoryID"]
aligned_df["trajectoryCv2"] = withIDCv2Sys["trajectoryID"]
aligned_df["trajectorySys"] = IdSysSys

# aligned_df.to_excel("data/ablation0_NoThing.xlsx", index=False)


In [14]:
fig     = go.Figure()
fig.add_trace(go.Scatter(
                                                x   = aligned_df["Time"],
                                                y   = aligned_df["OriginalHICr2"],
                                                mode= 'lines',
                                                name= "OriginalHICr1"
        ))

fig.update_layout(
                                    title       = "HI System",
                                    xaxis_title = "Time (arbitrary units)",
                                    yaxis_title = "Signal Value",
                                    template    = "plotly_white",
                                    showlegend  = True  # Show legend to allow toggling each signal
    )
    # Display the plot
fig.show()

In [15]:
# fig = {}
# for idxResults in aligned_df.columns[1:]:
#     fig[idxResults]     = go.Figure()
#     current_time        = 0

#     for idx, signal in enumerate(results[idxResults]):
#         signal          = tf.squeeze(signal).numpy()
#         time_vector     = np.arange(current_time, current_time + len(signal))
#         current_time    += len(signal)
        
#         # Add each signal as a separate trace
#         fig[idxResults].add_trace(go.Scatter(
#                                                 x   = time_vector,
#                                                 y   = signal,
#                                                 mode= 'lines',
#                                                 name= f'Signal {idx + 1}'  # Label each signal for legend toggling
#         ))
#     # Customize layout for readability
#     fig[idxResults].update_layout(
#                                     title       = idxResults,
#                                     xaxis_title = "Time (arbitrary units)",
#                                     yaxis_title = "Signal Value",
#                                     template    = "plotly_white",
#                                     showlegend  = True  # Show legend to allow toggling each signal
#     )
#     # Display the plot
#     fig[idxResults].show()

In [16]:
# Plotting the data with Plotly
# import plotly.express as px

# fig = px.line(df_result, x="Time", y=["System_HI"], 

#               labels={"value": "Health Index", "Time": "Time (s)"}, 

#               title="Health Index Over Time")

# fig.update_layout(
#     plot_bgcolor='white',  # Background of the plot area
#     paper_bgcolor='white' # Background of the overall canvas
# )

# fig.show()

In [17]:
# # Plotting the data with Plotly
# import plotly.express as px

# fig = px.line(df_result, x="Time", y=["OriginalHICr1", "OriginalHICr2", "OriginalHICv2"], 

#               labels={"value": "Health Index", "Time": "Time (s)"}, 

#               title="Health Index Over Time")



# fig.show()

In [18]:
# fig = {}
# for idxResults in aligned_df.columns[1:]:
#     fig[idxResults]     = go.Figure()
#     current_time        = 0

#     for idx, signal in enumerate(results[idxResults]):
#         signal          = tf.squeeze(signal).numpy()
#         time_vector     = np.arange(current_time, current_time + len(signal))
#         current_time    += len(signal)
        
#         # Add each signal as a separate trace
#         fig[idxResults].add_trace(go.Scatter(
#                                                 x   = time_vector,
#                                                 y   = signal,
#                                                 mode= 'lines',
#                                                 name= f'Signal {idx + 1}'  # Label each signal for legend toggling
#         ))
#     # Customize layout for readability
#     fig[idxResults].update_layout(
#                                     title       = idxResults,
#                                     xaxis_title = "Time (arbitrary units)",
#                                     yaxis_title = "Signal Value",
#                                     template    = "plotly_white",
#                                     showlegend  = True  # Show legend to allow toggling each signal
#     )
#     # Display the plot
#     fig[idxResults].show()

In [19]:
# Function to plot HI grouped by ID
# def plot_hi_by_id(time, hi, ids, title):
#     fig = go.Figure()
#     df_hi = pd.DataFrame({'Time': time, 'HI': hi, 'ID': ids})
#     grouped = df_hi.groupby('ID')
    
#     for id_val, group in grouped:
#         if not pd.isna(id_val):  # Skip NaN IDs
#             fig.add_trace(go.Scatter(
#                 x=group['Time'],
#                 y=group['HI'],
#                 mode='lines+markers',
#                 name=f'Trajectory {int(id_val)}'
#             ))
    
#     fig.update_layout(title=title, xaxis_title='Time', yaxis_title='Health Indicator (HI)')
#     return fig

# # Plot HI1, HI2, HI3
# withIDCr1Sys, withIDCr2Sys, withIDCv2Sys, IdSysSys
# ID1 = withIDCr1Sys["trajectoryID"]
# ID2 = withIDCr2Sys["trajectoryID"]
# ID3 = withIDCv2Sys["trajectoryID"]
# IDSys = IdSysSys


# fig_hi1 = plot_hi_by_id(aligned_df['Time'], aligned_df['HI1'], ID1, "Crusher 1 with Trajectories")
# fig_hi2 = plot_hi_by_id(aligned_df['Time'], aligned_df['HI2'], ID2, "Crusehr 2 with Trajectories")
# fig_hi3 = plot_hi_by_id(aligned_df['Time'], aligned_df['HI3'], ID3, "Conveyor betl with Trajectories")
# fig_hi4 = plot_hi_by_id(aligned_df['Time'], aligned_df['System_HI'], IDSys, "System_HI with Trajectories")

# # Plot System HI (no ID grouping)
# fig_system_hi = go.Figure()
# fig_system_hi.add_trace(go.Scatter(
#     x=aligned_df['Time'],
#     y=aligned_df['System_HI'],
#     mode='lines+markers',
#     name='System HI'
# ))
# fig_system_hi.update_layout(title="System HI", xaxis_title='Time', yaxis_title='System Health Indicator (HI)')

# # Display all plots
# fig_hi1.show()
# fig_hi2.show()
# fig_hi3.show()
# fig_system_hi.show()